## Install the required libraries

## Part-1 Configuration and data loading

### we can change the system configuration and target value this is main file

In [2]:
from __future__ import annotations

import gc
import json
import math
from pathlib import Path
from typing import Iterator

import joblib
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import torch

from sentence_transformers import SentenceTransformer
from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import IncrementalPCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm


# ============================================================
# CONFIGURATION
# ============================================================

CURRENT_DIRECTORY = Path.cwd()

PROJECT_ROOT = (
    CURRENT_DIRECTORY.parent
    if CURRENT_DIRECTORY.name.lower() == "notebooks"
    else CURRENT_DIRECTORY
)

AMAZON_INPUT_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "amazon"
    / "amazon_price_training.parquet"
)

CLUSTER_ROOT = (
    PROJECT_ROOT
    / "data"
    / "clustering"
    / "amazon"
)

FEATURE_ROOT = (
    CLUSTER_ROOT
    / "features"
)

MODEL_ROOT = (
    PROJECT_ROOT
    / "models"
    / "clustering"
)

REPORT_ROOT = (
    PROJECT_ROOT
    / "data"
    / "reports"
    / "clustering"
)

for folder in [
    CLUSTER_ROOT,
    FEATURE_ROOT,
    MODEL_ROOT,
    REPORT_ROOT,
]:
    folder.mkdir(
        parents=True,
        exist_ok=True,
    )


# ============================================================
# OUTPUT FILES
# ============================================================

TEXT_EMBEDDING_FILE = (
    FEATURE_ROOT
    / "amazon_text_embeddings.npy"
)

TEXT_INDEX_FILE = (
    FEATURE_ROOT
    / "amazon_text_embedding_index.parquet"
)

PCA_FEATURE_FILE = (
    FEATURE_ROOT
    / "amazon_text_pca.npy"
)

STRUCTURED_FEATURE_FILE = (
    FEATURE_ROOT
    / "amazon_structured_features.npy"
)

FINAL_CLUSTER_FEATURE_FILE = (
    FEATURE_ROOT
    / "amazon_cluster_features.npy"
)

CLUSTERED_DATASET_FILE = (
    CLUSTER_ROOT
    / "amazon_products_clustered.parquet"
)

REPRESENTATIVE_SAMPLE_FILE = (
    CLUSTER_ROOT
    / "amazon_representative_sample_20000.parquet"
)

CLUSTER_SUMMARY_FILE = (
    REPORT_ROOT
    / "cluster_summary.csv"
)

CLUSTER_SAMPLE_REPORT_FILE = (
    REPORT_ROOT
    / "cluster_sample_allocation.csv"
)

PCA_MODEL_FILE = (
    MODEL_ROOT
    / "incremental_pca.joblib"
)

SCALER_MODEL_FILE = (
    MODEL_ROOT
    / "structured_scaler.joblib"
)

KMEANS_MODEL_FILE = (
    MODEL_ROOT
    / "minibatch_kmeans.joblib"
)


# ============================================================
# PIPELINE SETTINGS FOR FULL 1.39M DATASET
# ============================================================

TEXT_MODEL_NAME = (
    "sentence-transformers/all-MiniLM-L6-v2"
)

TEXT_BATCH_SIZE = 512

PCA_BATCH_SIZE = 20_000
PCA_COMPONENTS = 64

N_CLUSTERS = 100
KMEANS_BATCH_SIZE = 20_000

TARGET_SAMPLE_SIZE = 20_000

RANDOM_STATE = 42

SILHOUETTE_SAMPLE_SIZE = 20_000




# ============================================================
# DEVICE
# ============================================================

if torch.cuda.is_available():
    DEVICE = "cuda"

elif (
    hasattr(torch.backends, "mps")
    and torch.backends.mps.is_available()
):
    DEVICE = "mps"

else:
    DEVICE = "cpu"


print("Project root:", PROJECT_ROOT)
print("Input file:", AMAZON_INPUT_FILE)
print("Device:", DEVICE)


# ============================================================
# INPUT VALIDATION
# ============================================================

if not AMAZON_INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Amazon dataset not found: "
        f"{AMAZON_INPUT_FILE}"
    )

print("✅ Amazon dataset found.")

Project root: /Users/souravkumar/Downloads/AI-Powered-Smart-Product-Pricing-Visual-Attribute-Extraction-System
Input file: /Users/souravkumar/Downloads/AI-Powered-Smart-Product-Pricing-Visual-Attribute-Extraction-System/data/processed/amazon/amazon_price_training.parquet
Device: mps
✅ Amazon dataset found.


## Part 2: Load and clean the clustering data

In [3]:
# ============================================================
# LOAD AMAZON DATA
# ============================================================

required_columns = [
    "asin",
    "title",
    "category_name",
    "price",
    "listPrice",
    "stars",
    "reviews",
    "isBestSeller",
    "boughtInLastMonth",
]

amazon_df = pd.read_parquet(
    AMAZON_INPUT_FILE,
    columns=required_columns,
)

print(
    "Rows loaded:",
    f"{len(amazon_df):,}",
)


# ============================================================
# CLEAN TEXT
# ============================================================

for column in [
    "asin",
    "title",
    "category_name",
]:
    amazon_df[column] = (
        amazon_df[column]
        .fillna("")
        .astype(str)
        .str.replace(
            r"\s+",
            " ",
            regex=True,
        )
        .str.strip()
    )


# ============================================================
# CLEAN NUMERIC FIELDS
# ============================================================

numeric_columns = [
    "price",
    "listPrice",
    "stars",
    "reviews",
    "boughtInLastMonth",
]

for column in numeric_columns:
    amazon_df[column] = pd.to_numeric(
        amazon_df[column],
        errors="coerce",
    )


amazon_df["isBestSeller"] = (
    amazon_df["isBestSeller"]
    .fillna(False)
    .astype(bool)
    .astype(np.int8)
)


# ============================================================
# FILTER INVALID ROWS
# ============================================================

amazon_df = amazon_df[
    amazon_df["asin"].ne("")
    & amazon_df["title"].ne("")
    & amazon_df["price"].notna()
    & amazon_df["price"].gt(0)
].copy()


amazon_df = amazon_df.drop_duplicates(
    subset=["asin"],
    keep="first",
)

amazon_df = amazon_df.reset_index(
    drop=True
)


# ============================================================
# FILL STRUCTURED VALUES
# ============================================================

amazon_df["category_name"] = (
    amazon_df["category_name"]
    .replace("", "Unknown")
)

amazon_df["listPrice"] = (
    amazon_df["listPrice"]
    .fillna(0)
    .clip(lower=0)
)

amazon_df["stars"] = (
    amazon_df["stars"]
    .fillna(0)
    .clip(lower=0, upper=5)
)

amazon_df["reviews"] = (
    amazon_df["reviews"]
    .fillna(0)
    .clip(lower=0)
)

amazon_df["boughtInLastMonth"] = (
    amazon_df["boughtInLastMonth"]
    .fillna(0)
    .clip(lower=0)
)


# ============================================================
# CREATE PRODUCT TEXT
# ============================================================

amazon_df["product_text"] = (
    "title: "
    + amazon_df["title"]
    + " | category: "
    + amazon_df["category_name"]
)


# ============================================================
# CREATE PRICE BANDS FOR LATER SAMPLING
# ============================================================

amazon_df["price_band"] = pd.qcut(
    amazon_df["price"],
    q=[
        0.00,
        0.20,
        0.40,
        0.60,
        0.80,
        0.95,
        1.00,
    ],
    labels=[
        "very_low",
        "low",
        "medium",
        "high",
        "premium",
        "luxury",
    ],
    duplicates="drop",
)


print(
    "Clean rows:",
    f"{len(amazon_df):,}",
)

print(
    "Categories:",
    f"{amazon_df['category_name'].nunique():,}",
)

display(
    amazon_df[
        [
            "asin",
            "title",
            "category_name",
            "price",
            "price_band",
            "product_text",
        ]
    ].head()
)

Rows loaded: 1,393,564
Clean rows: 1,393,564
Categories: 248


,asin,title,category_name,price,price_band,product_text
0,B014TMV5YE,"Sion Softside Expandable Roller Luggage, Black...",Suitcases,139.99,premium,title: Sion Softside Expandable Roller Luggage...
1,B07GDLCQXV,Luggage Sets Expandable PC+ABS Durable Suitcas...,Suitcases,169.99,luxury,title: Luggage Sets Expandable PC+ABS Durable ...
2,B07XSCCZYG,Platinum Elite Softside Expandable Checked Lug...,Suitcases,365.49,luxury,title: Platinum Elite Softside Expandable Chec...
3,B08MVFKGJM,Freeform Hardside Expandable with Double Spinn...,Suitcases,291.59,luxury,title: Freeform Hardside Expandable with Doubl...
4,B01DJLKZBA,Winfield 2 Hardside Expandable Luggage with Sp...,Suitcases,174.99,luxury,title: Winfield 2 Hardside Expandable Luggage ...


## Part 3: Generate text embeddings

### For 1.39 million rows, this can take time. The code writes embeddings into a NumPy memory-mapped file instead of storing everything in RAM.

In [4]:
# ============================================================
# LOAD SENTENCE TRANSFORMER
# ============================================================

print(
    "Loading text embedding model:",
    TEXT_MODEL_NAME,
)

text_model = SentenceTransformer(
    TEXT_MODEL_NAME,
    device=DEVICE,
)

embedding_dimension = (
    text_model
    .get_sentence_embedding_dimension()
)

print(
    "Embedding dimension:",
    embedding_dimension,
)


# ============================================================
# CREATE MEMORY-MAPPED EMBEDDING FILE
# ============================================================

number_of_products = len(
    amazon_df
)

text_embedding_memmap = np.lib.format.open_memmap(
    TEXT_EMBEDDING_FILE,
    mode="w+",
    dtype=np.float32,
    shape=(
        number_of_products,
        embedding_dimension,
    ),
)


# ============================================================
# GENERATE EMBEDDINGS IN BATCHES
# ============================================================

texts = amazon_df[
    "product_text"
].tolist()


for start_index in tqdm(
    range(
        0,
        number_of_products,
        TEXT_BATCH_SIZE,
    ),
    desc="Generating text embeddings",
):

    end_index = min(
        start_index + TEXT_BATCH_SIZE,
        number_of_products,
    )

    batch_texts = texts[
        start_index:end_index
    ]

    batch_embeddings = text_model.encode(
        batch_texts,
        batch_size=TEXT_BATCH_SIZE,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    text_embedding_memmap[
        start_index:end_index
    ] = batch_embeddings.astype(
        np.float32
    )


text_embedding_memmap.flush()


# ============================================================
# SAVE INDEX
# ============================================================

embedding_index_df = amazon_df[
    [
        "asin",
        "title",
        "category_name",
        "price",
        "price_band",
    ]
].copy()

embedding_index_df[
    "embedding_row"
] = np.arange(
    len(embedding_index_df)
)

embedding_index_df.to_parquet(
    TEXT_INDEX_FILE,
    index=False,
    compression="snappy",
)


print(
    "✅ Text embeddings saved:",
    TEXT_EMBEDDING_FILE,
)

print(
    "Embedding shape:",
    text_embedding_memmap.shape,
)

Loading text embedding model: sentence-transformers/all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding dimension: 384


/var/folders/m6/dqh8mk6d6szdv211v9m555xm0000gn/T/ipykernel_2113/169608061.py:17: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  .get_sentence_embedding_dimension()


Generating text embeddings:   0%|          | 0/2722 [00:00<?, ?it/s]

✅ Text embeddings saved: /Users/souravkumar/Downloads/AI-Powered-Smart-Product-Pricing-Visual-Attribute-Extraction-System/data/clustering/amazon/features/amazon_text_embeddings.npy
Embedding shape: (1393564, 384)


### Part 4: Create structured clustering features

In [8]:
# ============================================================
# CREATE STRUCTURED FEATURES
# ============================================================

structured_df = pd.DataFrame(
    {
        "price_log1p": np.log1p(
            amazon_df["price"]
        ),

        "list_price_log1p": np.log1p(
            amazon_df["listPrice"]
        ),

        "stars": amazon_df["stars"],

        "reviews_log1p": np.log1p(
            amazon_df["reviews"]
        ),

        "bought_log1p": np.log1p(
            amazon_df["boughtInLastMonth"]
        ),

        "is_best_seller": (
            amazon_df["isBestSeller"]
        ),
    }
)


# ============================================================
# SCALE STRUCTURED FEATURES
# ============================================================

structured_scaler = StandardScaler()

structured_features = (
    structured_scaler
    .fit_transform(
        structured_df
    )
    .astype(np.float32)
)


np.save(
    STRUCTURED_FEATURE_FILE,
    structured_features,
)

joblib.dump(
    structured_scaler,
    SCALER_MODEL_FILE,
)


print(
    "Structured feature shape:",
    structured_features.shape,
)

display(
    structured_df.head()
)

Structured feature shape: (1393564, 6)


,price_log1p,list_price_log1p,stars,reviews_log1p,bought_log1p,is_best_seller
0,4.948689,0.000000,4.5,0.0,7.601402,0
1,5.141605,5.351811,4.5,0.0,6.908755,0
2,5.903971,6.066085,4.6,0.0,5.707110,0
3,5.678772,5.873160,4.6,0.0,5.993961,0
4,5.170427,5.739761,4.5,0.0,5.993961,0


## Part 5: Reduce text embeddings using Incremental PCA

In [9]:
# ============================================================
# LOAD TEXT EMBEDDINGS AS MEMMAP
# ============================================================

text_embeddings = np.load(
    TEXT_EMBEDDING_FILE,
    mmap_mode="r",
)

print(
    "Text embedding shape:",
    text_embeddings.shape,
)


# ============================================================
# FIT INCREMENTAL PCA
# ============================================================

incremental_pca = IncrementalPCA(
    n_components=PCA_COMPONENTS,
    batch_size=PCA_BATCH_SIZE,
)


for start_index in tqdm(
    range(
        0,
        len(text_embeddings),
        PCA_BATCH_SIZE,
    ),
    desc="Fitting Incremental PCA",
):

    end_index = min(
        start_index + PCA_BATCH_SIZE,
        len(text_embeddings),
    )

    batch = np.asarray(
        text_embeddings[
            start_index:end_index
        ],
        dtype=np.float32,
    )

    # IncrementalPCA requires each partial-fit batch
    # to contain at least n_components rows.
    if len(batch) < PCA_COMPONENTS:
        break

    incremental_pca.partial_fit(
        batch
    )


joblib.dump(
    incremental_pca,
    PCA_MODEL_FILE,
)


# ============================================================
# TRANSFORM EMBEDDINGS
# ============================================================

pca_memmap = np.lib.format.open_memmap(
    PCA_FEATURE_FILE,
    mode="w+",
    dtype=np.float32,
    shape=(
        len(text_embeddings),
        PCA_COMPONENTS,
    ),
)


for start_index in tqdm(
    range(
        0,
        len(text_embeddings),
        PCA_BATCH_SIZE,
    ),
    desc="Transforming text embeddings",
):

    end_index = min(
        start_index + PCA_BATCH_SIZE,
        len(text_embeddings),
    )

    batch = np.asarray(
        text_embeddings[
            start_index:end_index
        ],
        dtype=np.float32,
    )

    transformed_batch = (
        incremental_pca
        .transform(batch)
        .astype(np.float32)
    )

    pca_memmap[
        start_index:end_index
    ] = transformed_batch


pca_memmap.flush()


print(
    "PCA output shape:",
    pca_memmap.shape,
)

print(
    "Explained variance:",
    float(
        incremental_pca
        .explained_variance_ratio_
        .sum()
    ),
)

Text embedding shape: (1393564, 384)


Fitting Incremental PCA:   0%|          | 0/70 [00:00<?, ?it/s]

Transforming text embeddings:   0%|          | 0/70 [00:00<?, ?it/s]

PCA output shape: (1393564, 64)
Explained variance: 0.6295153507096409


## Part 6: Combine semantic and structured features

In [10]:
# ============================================================
# FEATURE WEIGHTS
# ============================================================

TEXT_WEIGHT = 1.0
STRUCTURED_WEIGHT = 0.35


# ============================================================
# LOAD PCA FEATURES
# ============================================================

text_pca_features = np.load(
    PCA_FEATURE_FILE,
    mmap_mode="r",
)


# ============================================================
# CREATE FINAL FEATURE MATRIX
# ============================================================

final_feature_dimension = (
    text_pca_features.shape[1]
    + structured_features.shape[1]
)


cluster_feature_memmap = (
    np.lib.format.open_memmap(
        FINAL_CLUSTER_FEATURE_FILE,
        mode="w+",
        dtype=np.float32,
        shape=(
            len(amazon_df),
            final_feature_dimension,
        ),
    )
)


for start_index in tqdm(
    range(
        0,
        len(amazon_df),
        PCA_BATCH_SIZE,
    ),
    desc="Combining clustering features",
):

    end_index = min(
        start_index + PCA_BATCH_SIZE,
        len(amazon_df),
    )

    semantic_batch = (
        np.asarray(
            text_pca_features[
                start_index:end_index
            ],
            dtype=np.float32,
        )
        * TEXT_WEIGHT
    )

    structured_batch = (
        structured_features[
            start_index:end_index
        ]
        * STRUCTURED_WEIGHT
    )

    combined_batch = np.concatenate(
        [
            semantic_batch,
            structured_batch,
        ],
        axis=1,
    )

    cluster_feature_memmap[
        start_index:end_index
    ] = combined_batch


cluster_feature_memmap.flush()


print(
    "Final clustering feature shape:",
    cluster_feature_memmap.shape,
)

Combining clustering features:   0%|          | 0/70 [00:00<?, ?it/s]

Final clustering feature shape: (1393564, 70)


## Part 7: Train MiniBatch K-Means

In [11]:
# ============================================================
# LOAD CLUSTER FEATURES
# ============================================================

cluster_features = np.load(
    FINAL_CLUSTER_FEATURE_FILE,
    mmap_mode="r",
)


# ============================================================
# TRAIN MINI-BATCH K-MEANS
# ============================================================

kmeans_model = MiniBatchKMeans(
    n_clusters=N_CLUSTERS,
    batch_size=KMEANS_BATCH_SIZE,
    random_state=RANDOM_STATE,
    n_init="auto",
    reassignment_ratio=0.01,
    max_no_improvement=20,
    verbose=0,
)


kmeans_model.fit(
    cluster_features
)


joblib.dump(
    kmeans_model,
    KMEANS_MODEL_FILE,
)


print(
    "✅ MiniBatchKMeans training complete."
)

print(
    "Model inertia:",
    kmeans_model.inertia_,
)

✅ MiniBatchKMeans training complete.
Model inertia: 623244.8125


### Part 8: Assign cluster IDs

In [12]:
# ============================================================
# PREDICT CLUSTERS IN BATCHES
# ============================================================

cluster_ids = np.empty(
    len(amazon_df),
    dtype=np.int32,
)


for start_index in tqdm(
    range(
        0,
        len(amazon_df),
        KMEANS_BATCH_SIZE,
    ),
    desc="Assigning cluster IDs",
):

    end_index = min(
        start_index + KMEANS_BATCH_SIZE,
        len(amazon_df),
    )

    cluster_ids[
        start_index:end_index
    ] = kmeans_model.predict(
        cluster_features[
            start_index:end_index
        ]
    )


amazon_df["cluster_id"] = (
    cluster_ids
)


amazon_df.to_parquet(
    CLUSTERED_DATASET_FILE,
    index=False,
    compression="snappy",
)


print(
    "✅ Clustered dataset saved:",
    CLUSTERED_DATASET_FILE,
)

print(
    "Clusters found:",
    amazon_df["cluster_id"].nunique(),
)

Assigning cluster IDs:   0%|          | 0/70 [00:00<?, ?it/s]

✅ Clustered dataset saved: /Users/souravkumar/Downloads/AI-Powered-Smart-Product-Pricing-Visual-Attribute-Extraction-System/data/clustering/amazon/amazon_products_clustered.parquet
Clusters found: 100


## Part 9: Validate cluster quality

In [13]:
# ============================================================
# CLUSTER SIZE DISTRIBUTION
# ============================================================

cluster_counts = (
    amazon_df["cluster_id"]
    .value_counts()
    .sort_index()
)


cluster_summary_df = pd.DataFrame(
    {
        "cluster_id": cluster_counts.index,
        "product_count": cluster_counts.values,
    }
)


cluster_summary_df[
    "percentage"
] = (
    cluster_summary_df[
        "product_count"
    ]
    / len(amazon_df)
    * 100
)


# ============================================================
# TOP CATEGORY PER CLUSTER
# ============================================================

top_categories = (
    amazon_df
    .groupby(
        [
            "cluster_id",
            "category_name",
        ]
    )
    .size()
    .reset_index(
        name="count"
    )
    .sort_values(
        [
            "cluster_id",
            "count",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .groupby(
        "cluster_id"
    )
    .head(1)
    .rename(
        columns={
            "category_name": (
                "top_category"
            ),
            "count": (
                "top_category_count"
            ),
        }
    )
)


cluster_summary_df = (
    cluster_summary_df
    .merge(
        top_categories[
            [
                "cluster_id",
                "top_category",
                "top_category_count",
            ]
        ],
        on="cluster_id",
        how="left",
    )
)


# ============================================================
# PRICE STATISTICS PER CLUSTER
# ============================================================

price_stats = (
    amazon_df
    .groupby(
        "cluster_id"
    )["price"]
    .agg(
        [
            "min",
            "median",
            "mean",
            "max",
        ]
    )
    .reset_index()
    .rename(
        columns={
            "min": "minimum_price",
            "median": "median_price",
            "mean": "mean_price",
            "max": "maximum_price",
        }
    )
)


cluster_summary_df = (
    cluster_summary_df
    .merge(
        price_stats,
        on="cluster_id",
        how="left",
    )
)


# ============================================================
# SILHOUETTE SCORE
# ============================================================

silhouette_size = min(
    SILHOUETTE_SAMPLE_SIZE,
    len(amazon_df),
)


silhouette_indices = np.random.default_rng(
    RANDOM_STATE
).choice(
    len(amazon_df),
    size=silhouette_size,
    replace=False,
)


silhouette_features = np.asarray(
    cluster_features[
        silhouette_indices
    ],
    dtype=np.float32,
)


silhouette_labels = cluster_ids[
    silhouette_indices
]


silhouette_value = silhouette_score(
    silhouette_features,
    silhouette_labels,
    metric="euclidean",
)


cluster_summary_df.to_csv(
    CLUSTER_SUMMARY_FILE,
    index=False,
)


print(
    "Silhouette score:",
    round(
        float(silhouette_value),
        4,
    ),
)

display(
    cluster_summary_df.head(20)
)

Silhouette score: 0.0664


,cluster_id,product_count,percentage,top_category,top_category_count,minimum_price,median_price,mean_price,maximum_price
0,0,16939,1.215516,Boys' Clothing,3051,3.88,22.79,24.451518,80.00
1,1,24194,1.736124,"Lights, Bulbs & Indicators",3010,1.99,17.97,19.184328,54.99
2,2,7463,0.535533,Men's Clothing,1090,3.47,18.99,22.083992,93.00
3,3,1445,0.103691,Automotive Tools & Equipment,138,0.99,18.79,28.606173,549.00
4,4,17626,1.264815,Industrial Materials,2425,0.01,9.99,11.062749,32.97
5,5,14840,1.064895,Sports & Outdoor Play Toys,1463,7.99,29.99,37.817181,387.99
6,6,20879,1.498245,Wall Art,4676,2.99,19.99,23.600760,89.99
7,7,15728,1.128617,Televisions & Video Products,2797,1.79,16.99,19.727674,124.99
8,8,14826,1.063891,Toys & Games,2325,2.64,17.49,19.393353,63.91
9,9,18161,1.303205,Automotive Performance Parts & Accessories,2823,54.27,139.00,159.384091,629.06


## Part 10: Create the representative 20,000-product sample

In [14]:
# ============================================================
# SAMPLE ALLOCATION SETTINGS
# ============================================================

MINIMUM_PER_CLUSTER = 50
MAXIMUM_PER_CLUSTER = 500


cluster_sizes = (
    amazon_df["cluster_id"]
    .value_counts()
    .sort_index()
)


total_products = len(
    amazon_df
)


# ============================================================
# INITIAL ALLOCATION
# ============================================================

allocation_rows = []


for cluster_id, cluster_size in (
    cluster_sizes.items()
):

    proportional_allocation = int(
        round(
            TARGET_SAMPLE_SIZE
            * cluster_size
            / total_products
        )
    )

    allocated = max(
        MINIMUM_PER_CLUSTER,
        proportional_allocation,
    )

    allocated = min(
        allocated,
        MAXIMUM_PER_CLUSTER,
        cluster_size,
    )

    allocation_rows.append(
        {
            "cluster_id": cluster_id,
            "cluster_size": cluster_size,
            "allocated_samples": allocated,
        }
    )


allocation_df = pd.DataFrame(
    allocation_rows
)


# ============================================================
# ADJUST ALLOCATION TO TARGET SIZE
# ============================================================

current_allocation = int(
    allocation_df[
        "allocated_samples"
    ].sum()
)


while current_allocation > TARGET_SAMPLE_SIZE:

    adjustable = allocation_df[
        allocation_df[
            "allocated_samples"
        ] > MINIMUM_PER_CLUSTER
    ]

    if adjustable.empty:
        break

    index_to_reduce = (
        adjustable[
            "allocated_samples"
        ]
        .idxmax()
    )

    allocation_df.loc[
        index_to_reduce,
        "allocated_samples",
    ] -= 1

    current_allocation -= 1


while current_allocation < TARGET_SAMPLE_SIZE:

    adjustable = allocation_df[
        (
            allocation_df[
                "allocated_samples"
            ]
            < allocation_df[
                "cluster_size"
            ]
        )
        & (
            allocation_df[
                "allocated_samples"
            ]
            < MAXIMUM_PER_CLUSTER
        )
    ]

    if adjustable.empty:
        break

    index_to_increase = (
        adjustable[
            "cluster_size"
        ]
        .idxmax()
    )

    allocation_df.loc[
        index_to_increase,
        "allocated_samples",
    ] += 1

    current_allocation += 1


print(
    "Final allocated sample size:",
    current_allocation,
)

Final allocated sample size: 20000


In [15]:
# ============================================================
# CLUSTER-BASED REPRESENTATIVE SAMPLING
# ============================================================

sample_parts = []


for allocation in allocation_df.itertuples(
    index=False
):

    cluster_id = allocation.cluster_id

    sample_count = (
        allocation.allocated_samples
    )

    cluster_df = amazon_df[
        amazon_df["cluster_id"].eq(
            cluster_id
        )
    ].copy()


    if len(cluster_df) <= sample_count:
        selected_df = cluster_df

    else:
        price_band_groups = [
            group
            for _, group
            in cluster_df.groupby(
                "price_band",
                observed=True,
            )
        ]


        selected_parts = []

        if price_band_groups:
            base_per_price_band = max(
                1,
                sample_count
                // len(price_band_groups),
            )

            for price_group in (
                price_band_groups
            ):
                count = min(
                    len(price_group),
                    base_per_price_band,
                )

                selected_parts.append(
                    price_group.sample(
                        n=count,
                        random_state=(
                            RANDOM_STATE
                            + int(cluster_id)
                        ),
                    )
                )


        selected_df = pd.concat(
            selected_parts,
            ignore_index=False,
        )


        selected_df = (
            selected_df
            .drop_duplicates(
                subset=["asin"]
            )
        )


        remaining_quota = (
            sample_count
            - len(selected_df)
        )


        if remaining_quota > 0:

            remaining_cluster_df = (
                cluster_df[
                    ~cluster_df[
                        "asin"
                    ].isin(
                        selected_df[
                            "asin"
                        ]
                    )
                ]
            )

            additional_count = min(
                remaining_quota,
                len(
                    remaining_cluster_df
                ),
            )

            if additional_count > 0:
                additional_df = (
                    remaining_cluster_df
                    .sample(
                        n=additional_count,
                        random_state=(
                            RANDOM_STATE
                            + int(cluster_id)
                            + 1000
                        ),
                    )
                )

                selected_df = pd.concat(
                    [
                        selected_df,
                        additional_df,
                    ],
                    ignore_index=False,
                )


    sample_parts.append(
        selected_df
    )


representative_sample_df = pd.concat(
    sample_parts,
    ignore_index=True,
)


representative_sample_df = (
    representative_sample_df
    .drop_duplicates(
        subset=["asin"]
    )
    .reset_index(drop=True)
)


# ============================================================
# FINAL QUOTA CORRECTION
# ============================================================

if (
    len(representative_sample_df)
    < TARGET_SAMPLE_SIZE
):

    selected_asins = set(
        representative_sample_df[
            "asin"
        ]
    )

    remaining_df = amazon_df[
        ~amazon_df["asin"].isin(
            selected_asins
        )
    ]

    additional_size = min(
        TARGET_SAMPLE_SIZE
        - len(
            representative_sample_df
        ),
        len(remaining_df),
    )

    additional_df = remaining_df.sample(
        n=additional_size,
        random_state=RANDOM_STATE + 9999,
    )

    representative_sample_df = (
        pd.concat(
            [
                representative_sample_df,
                additional_df,
            ],
            ignore_index=True,
        )
    )


if (
    len(representative_sample_df)
    > TARGET_SAMPLE_SIZE
):
    representative_sample_df = (
        representative_sample_df
        .sample(
            n=TARGET_SAMPLE_SIZE,
            random_state=RANDOM_STATE,
        )
        .reset_index(drop=True)
    )


representative_sample_df.to_parquet(
    REPRESENTATIVE_SAMPLE_FILE,
    index=False,
    compression="snappy",
)


allocation_df.to_csv(
    CLUSTER_SAMPLE_REPORT_FILE,
    index=False,
)


print(
    "✅ Representative sample created:",
    f"{len(representative_sample_df):,}",
)

print(
    "Clusters represented:",
    representative_sample_df[
        "cluster_id"
    ].nunique(),
)

print(
    "Categories represented:",
    representative_sample_df[
        "category_name"
    ].nunique(),
)

display(
    representative_sample_df[
        [
            "asin",
            "title",
            "category_name",
            "price",
            "price_band",
            "cluster_id",
        ]
    ].head(20)
)

✅ Representative sample created: 20,000
Clusters represented: 100
Categories represented: 242


,asin,title,category_name,price,price_band,cluster_id
0,B093JZHY6P,Girls' Zombies Underwear Multipack,Girls' Clothing,10.56,very_low,0
1,B09M9ZDQR6,Infant Boys and Girls Oxford Shoes PU Leather ...,Baby Boys' Clothing & Shoes,8.39,very_low,0
2,B002U8280U,Unisex-Baby Cozy Fleece Booties Slipper Sock,Baby Boys' Clothing & Shoes,10.00,very_low,0
3,B093PVSNB8,Kids Water Sports Shoes Ultra Light Totally Dr...,Girls' Shoes,9.99,very_low,0
4,B07X6BSRPT,"DUMBO unisex baby Dumbo 5 Pack Shorty Socks, A...",Baby Boys' Clothing & Shoes,8.47,very_low,0
5,B0BL5X14NC,Pokemon Pkmn Team Ghost Group Girls Short Slee...,Girls' Clothing,5.67,very_low,0
6,B00X243QMK,Luvable Friends Baby Socks Set,Baby Boys' Clothing & Shoes,7.46,very_low,0
7,B0BWDNGQ32,"Ballerina House Slippers for Women, Anti-Skid ...",Women's Shoes,9.49,very_low,0
8,B06WVM42MB,Baby Boys Swimming Trunks Cartoon Swimwear Sho...,Boys' Clothing,9.99,very_low,0
9,B0BM8ZRXZH,Girls' and Turn Cuff Socks,Girls' Clothing,6.90,very_low,0


## Step 1: Validate the 20,000-product sample

In [17]:
from pathlib import Path

import pandas as pd


# ============================================================
# PATH CONFIGURATION
# ============================================================

CURRENT_DIRECTORY = Path.cwd()

PROJECT_ROOT = (
    CURRENT_DIRECTORY.parent
    if CURRENT_DIRECTORY.name.lower() == "notebooks"
    else CURRENT_DIRECTORY
)

ORIGINAL_SAMPLE_FILE = (
    PROJECT_ROOT
    / "data"
    / "clustering"
    / "amazon"
    / "amazon_representative_sample_20000.parquet"
)

FIXED_SAMPLE_FILE = (
    PROJECT_ROOT
    / "data"
    / "clustering"
    / "amazon"
    / "amazon_representative_sample_20000_with_images.parquet"
)

AMAZON_SOURCE_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "amazon"
    / "amazon_price_training.parquet"
)


# ============================================================
# DISPLAY HELPERS
# ============================================================

def print_header(title: str) -> None:
    print()
    print("=" * 80)
    print(title)
    print("=" * 80)


def print_pass(message: str) -> None:
    print(f"✅ [PASS] {message}")


def print_warning(message: str) -> None:
    print(f"⚠️  [WARNING] {message}")


def print_fail(message: str) -> None:
    print(f"❌ [FAIL] {message}")


# ============================================================
# LOAD REPRESENTATIVE SAMPLE
# ============================================================

print_header("LOADING REPRESENTATIVE SAMPLE")

if FIXED_SAMPLE_FILE.exists():
    SAMPLE_FILE = FIXED_SAMPLE_FILE
    print_pass(
        "Using representative sample containing image URLs."
    )

elif ORIGINAL_SAMPLE_FILE.exists():
    SAMPLE_FILE = ORIGINAL_SAMPLE_FILE
    print_warning(
        "Fixed sample was not found. "
        "Loading the original sample."
    )

else:
    raise FileNotFoundError(
        "Neither representative sample file was found.\n"
        f"Original: {ORIGINAL_SAMPLE_FILE}\n"
        f"Fixed: {FIXED_SAMPLE_FILE}"
    )


sample_df = pd.read_parquet(
    SAMPLE_FILE
)

print("Sample file:", SAMPLE_FILE)
print("Rows loaded:", f"{len(sample_df):,}")


# ============================================================
# NORMALIZE ASIN
# ============================================================

if "asin" not in sample_df.columns:
    raise ValueError(
        "The representative sample does not contain the 'asin' column."
    )

sample_df["asin"] = (
    sample_df["asin"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.upper()
)


# ============================================================
# RESTORE IMAGE URL IF MISSING
# ============================================================

if "imgUrl" not in sample_df.columns:

    print_warning(
        "'imgUrl' is missing from the sample. "
        "Joining it from the Amazon price-training dataset."
    )

    if not AMAZON_SOURCE_FILE.exists():
        raise FileNotFoundError(
            f"Amazon source file not found: {AMAZON_SOURCE_FILE}"
        )

    amazon_image_df = pd.read_parquet(
        AMAZON_SOURCE_FILE,
        columns=[
            "asin",
            "imgUrl",
            "productURL",
        ],
    )

    amazon_image_df["asin"] = (
        amazon_image_df["asin"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.upper()
    )

    amazon_image_df = (
        amazon_image_df
        .drop_duplicates(
            subset=["asin"],
            keep="first",
        )
    )

    sample_df = sample_df.merge(
        amazon_image_df,
        on="asin",
        how="left",
    )

    sample_df.to_parquet(
        FIXED_SAMPLE_FILE,
        index=False,
        compression="snappy",
    )

    SAMPLE_FILE = FIXED_SAMPLE_FILE

    print_pass(
        f"Image URLs restored and fixed file saved:\n{FIXED_SAMPLE_FILE}"
    )


# ============================================================
# CLEAN IMPORTANT COLUMNS
# ============================================================

text_columns = [
    "title",
    "imgUrl",
    "category_name",
    "price_band",
]

for column in text_columns:
    if column in sample_df.columns:
        sample_df[column] = (
            sample_df[column]
            .fillna("")
            .astype(str)
            .str.replace(
                r"\s+",
                " ",
                regex=True,
            )
            .str.strip()
        )


if "price" in sample_df.columns:
    sample_df["price"] = pd.to_numeric(
        sample_df["price"],
        errors="coerce",
    )


if "cluster_id" in sample_df.columns:
    sample_df["cluster_id"] = pd.to_numeric(
        sample_df["cluster_id"],
        errors="coerce",
    )


# ============================================================
# BASIC DATASET SUMMARY
# ============================================================

print_header("REPRESENTATIVE SAMPLE SUMMARY")

print("Rows:", f"{len(sample_df):,}")
print(
    "Unique ASINs:",
    f"{sample_df['asin'].nunique():,}",
)

if "cluster_id" in sample_df.columns:
    print(
        "Clusters:",
        sample_df["cluster_id"].nunique(),
    )

if "category_name" in sample_df.columns:
    print(
        "Categories:",
        sample_df["category_name"].nunique(),
    )

if "price_band" in sample_df.columns:
    print(
        "Price bands:",
        sample_df["price_band"].nunique(),
    )


# ============================================================
# REQUIRED COLUMN VALIDATION
# ============================================================

print_header("REQUIRED COLUMN VALIDATION")

required_columns = [
    "asin",
    "title",
    "imgUrl",
    "price",
    "category_name",
    "price_band",
    "cluster_id",
]

missing_required_columns = []

for column in required_columns:

    if column not in sample_df.columns:
        print_fail(
            f"Required column missing: {column}"
        )

        missing_required_columns.append(
            column
        )

    else:
        missing_count = int(
            sample_df[column]
            .isna()
            .sum()
        )

        empty_count = 0

        if (
            pd.api.types.is_object_dtype(
                sample_df[column]
            )
            or pd.api.types.is_string_dtype(
                sample_df[column]
            )
        ):
            empty_count = int(
                sample_df[column]
                .fillna("")
                .astype(str)
                .str.strip()
                .eq("")
                .sum()
            )

        print_pass(
            f"{column}: "
            f"{missing_count:,} missing, "
            f"{empty_count:,} empty"
        )


if missing_required_columns:
    raise ValueError(
        "Required columns are still missing: "
        + ", ".join(
            missing_required_columns
        )
    )


# ============================================================
# DATA QUALITY CHECKS
# ============================================================

print_header("DATA QUALITY CHECKS")


duplicate_asin_count = int(
    sample_df["asin"]
    .duplicated()
    .sum()
)

invalid_price_count = int(
    (
        sample_df["price"].isna()
        | sample_df["price"].le(0)
    ).sum()
)

valid_image_url_mask = (
    sample_df["imgUrl"]
    .fillna("")
    .astype(str)
    .str.startswith(
        ("http://", "https://")
    )
)

invalid_image_url_count = int(
    (~valid_image_url_mask).sum()
)

missing_title_count = int(
    sample_df["title"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

missing_category_count = int(
    sample_df["category_name"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

missing_cluster_count = int(
    sample_df["cluster_id"]
    .isna()
    .sum()
)


print(
    "Duplicate ASINs:",
    f"{duplicate_asin_count:,}",
)

print(
    "Invalid prices:",
    f"{invalid_price_count:,}",
)

print(
    "Invalid or missing image URLs:",
    f"{invalid_image_url_count:,}",
)

print(
    "Missing titles:",
    f"{missing_title_count:,}",
)

print(
    "Missing categories:",
    f"{missing_category_count:,}",
)

print(
    "Missing cluster IDs:",
    f"{missing_cluster_count:,}",
)


# ============================================================
# OVERALL VALIDATION RESULT
# ============================================================

quality_issues = {
    "duplicate_asins": duplicate_asin_count,
    "invalid_prices": invalid_price_count,
    "invalid_image_urls": invalid_image_url_count,
    "missing_titles": missing_title_count,
    "missing_categories": missing_category_count,
    "missing_cluster_ids": missing_cluster_count,
}

total_quality_issues = sum(
    quality_issues.values()
)

if total_quality_issues == 0:
    print_pass(
        "Representative sample passed all primary quality checks."
    )
else:
    print_warning(
        f"Representative sample contains "
        f"{total_quality_issues:,} total quality issues."
    )


# ============================================================
# DISTRIBUTION REPORTS
# ============================================================

print_header("CLUSTER DISTRIBUTION")

cluster_distribution = (
    sample_df["cluster_id"]
    .value_counts()
    .sort_index()
)

display(
    cluster_distribution.describe()
)

display(
    cluster_distribution.head(20)
)


print_header("PRICE-BAND DISTRIBUTION")

price_band_distribution = (
    sample_df["price_band"]
    .value_counts()
)

display(
    price_band_distribution
)


print_header("TOP 20 CATEGORIES")

top_categories = (
    sample_df["category_name"]
    .value_counts()
    .head(20)
)

display(
    top_categories
)


print_header("CATEGORY × PRICE BAND")

category_price_table = pd.crosstab(
    sample_df["category_name"],
    sample_df["price_band"],
)

display(
    category_price_table.head(20)
)


# ============================================================
# PRICE STATISTICS
# ============================================================

print_header("PRICE STATISTICS")

display(
    sample_df["price"]
    .describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
        ]
    )
)


# ============================================================
# FINAL SAMPLE PREVIEW
# ============================================================

print_header("REPRESENTATIVE SAMPLE PREVIEW")

preview_columns = [
    "asin",
    "title",
    "category_name",
    "price",
    "price_band",
    "cluster_id",
    "imgUrl",
]

display(
    sample_df[
        preview_columns
    ].head(20)
)


# ============================================================
# FINAL RESULT
# ============================================================

print_header("REPRESENTATIVE SAMPLE VALIDATION COMPLETED")

print("Validated file:", SAMPLE_FILE)
print("Rows:", f"{len(sample_df):,}")
print(
    "Valid image URLs:",
    f"{valid_image_url_mask.sum():,}",
)

print(
    "Invalid image URLs:",
    f"{invalid_image_url_count:,}",
)


LOADING REPRESENTATIVE SAMPLE
⚠️  [WARNING] Fixed sample was not found. Loading the original sample.
Sample file: /Users/souravkumar/Downloads/AI-Powered-Smart-Product-Pricing-Visual-Attribute-Extraction-System/data/clustering/amazon/amazon_representative_sample_20000.parquet
Rows loaded: 20,000
⚠️  [WARNING] 'imgUrl' is missing from the sample. Joining it from the Amazon price-training dataset.
✅ [PASS] Image URLs restored and fixed file saved:
/Users/souravkumar/Downloads/AI-Powered-Smart-Product-Pricing-Visual-Attribute-Extraction-System/data/clustering/amazon/amazon_representative_sample_20000_with_images.parquet

REPRESENTATIVE SAMPLE SUMMARY
Rows: 20,000
Unique ASINs: 20,000
Clusters: 100
Categories: 242
Price bands: 6

REQUIRED COLUMN VALIDATION
✅ [PASS] asin: 0 missing, 0 empty
✅ [PASS] title: 0 missing, 0 empty
✅ [PASS] imgUrl: 0 missing, 0 empty
✅ [PASS] price: 0 missing, 0 empty
✅ [PASS] category_name: 0 missing, 0 empty
✅ [PASS] price_band: 0 missing, 0 empty
✅ [PASS] clus

count    100.00000
mean     200.00000
std       86.19276
min       50.00000
25%      139.25000
50%      194.50000
75%      242.25000
max      470.00000
Name: count, dtype: float64

cluster_id
0     243
1     347
2     107
3      50
4     253
5     213
6     300
7     226
8     213
9     261
10    102
11    102
12    234
13     84
14    264
15     69
16    164
17    193
18    306
19    203
Name: count, dtype: int64


PRICE-BAND DISTRIBUTION


price_band
high        4018
medium      3921
low         3720
premium     3490
very_low    3420
luxury      1431
Name: count, dtype: int64

...


PRICE STATISTICS


count    20000.000000
mean        46.611362
std        121.720526
min          0.680000
25%         13.235000
50%         21.990000
75%         43.577500
90%         89.990000
95%        165.990000
99%        399.990000
max       8499.990000
Name: price, dtype: float64


REPRESENTATIVE SAMPLE PREVIEW


,asin,title,category_name,price,price_band,cluster_id,imgUrl
0,B093JZHY6P,Girls' Zombies Underwear Multipack,Girls' Clothing,10.56,very_low,0,https://m.media-amazon.com/images/I/91K1fyuzZd...
1,B09M9ZDQR6,Infant Boys and Girls Oxford Shoes PU Leather ...,Baby Boys' Clothing & Shoes,8.39,very_low,0,https://m.media-amazon.com/images/I/610FrPmAcJ...
2,B002U8280U,Unisex-Baby Cozy Fleece Booties Slipper Sock,Baby Boys' Clothing & Shoes,10.00,very_low,0,https://m.media-amazon.com/images/I/71lzQGcICc...
3,B093PVSNB8,Kids Water Sports Shoes Ultra Light Totally Dr...,Girls' Shoes,9.99,very_low,0,https://m.media-amazon.com/images/I/8106GfMldM...
4,B07X6BSRPT,"DUMBO unisex baby Dumbo 5 Pack Shorty Socks, A...",Baby Boys' Clothing & Shoes,8.47,very_low,0,https://m.media-amazon.com/images/I/9183pnt6x1...
5,B0BL5X14NC,Pokemon Pkmn Team Ghost Group Girls Short Slee...,Girls' Clothing,5.67,very_low,0,https://m.media-amazon.com/images/I/61+vbmQf1l...
6,B00X243QMK,Luvable Friends Baby Socks Set,Baby Boys' Clothing & Shoes,7.46,very_low,0,https://m.media-amazon.com/images/I/71wK3C1iTn...
7,B0BWDNGQ32,"Ballerina House Slippers for Women, Anti-Skid ...",Women's Shoes,9.49,very_low,0,https://m.media-amazon.com/images/I/7147PYuXH-...
8,B06WVM42MB,Baby Boys Swimming Trunks Cartoon Swimwear Sho...,Boys' Clothing,9.99,very_low,0,https://m.media-amazon.com/images/I/61kyelq8AL...
9,B0BM8ZRXZH,Girls' and Turn Cuff Socks,Girls' Clothing,6.90,very_low,0,https://m.media-amazon.com/images/I/61ugGBHLr5...



REPRESENTATIVE SAMPLE VALIDATION COMPLETED
Validated file: /Users/souravkumar/Downloads/AI-Powered-Smart-Product-Pricing-Visual-Attribute-Extraction-System/data/clustering/amazon/amazon_representative_sample_20000_with_images.parquet
Rows: 20,000
Valid image URLs: 20,000
Invalid image URLs: 0


## Next step: download the 20,000 images

In [18]:
from __future__ import annotations

import concurrent.futures
import hashlib
import time
from pathlib import Path
from typing import Any

import pandas as pd
import requests
from PIL import Image, ImageFile
from tqdm.auto import tqdm


# ============================================================
# CONFIGURATION
# ============================================================

CURRENT_DIRECTORY = Path.cwd()

PROJECT_ROOT = (
    CURRENT_DIRECTORY.parent
    if CURRENT_DIRECTORY.name.lower() == "notebooks"
    else CURRENT_DIRECTORY
)

SAMPLE_FILE = (
    PROJECT_ROOT
    / "data"
    / "clustering"
    / "amazon"
    / "amazon_representative_sample_20000_with_images.parquet"
)

OUTPUT_ROOT = (
    PROJECT_ROOT
    / "data"
    / "amazon_multimodal"
)

IMAGE_ROOT = OUTPUT_ROOT / "images"
REPORT_ROOT = OUTPUT_ROOT / "reports"
MODEL_INPUT_ROOT = OUTPUT_ROOT / "model_input"

for folder in [
    IMAGE_ROOT,
    REPORT_ROOT,
    MODEL_INPUT_ROOT,
]:
    folder.mkdir(
        parents=True,
        exist_ok=True,
    )

DOWNLOAD_REPORT_FILE = (
    REPORT_ROOT
    / "image_download_report.csv"
)

USABLE_DATASET_FILE = (
    MODEL_INPUT_ROOT
    / "amazon_multimodal_usable.parquet"
)

MAX_WORKERS = 12
REQUEST_TIMEOUT = 20
MAX_RETRIES = 3

ImageFile.LOAD_TRUNCATED_IMAGES = True


# ============================================================
# HELPERS
# ============================================================

def build_image_path(
    asin: str,
    image_url: str,
) -> Path:

    url_hash = hashlib.md5(
        image_url.encode("utf-8")
    ).hexdigest()[:10]

    image_folder = (
        IMAGE_ROOT
        / asin[:2].upper()
    )

    image_folder.mkdir(
        parents=True,
        exist_ok=True,
    )

    return (
        image_folder
        / f"{asin}_{url_hash}.jpg"
    )


def validate_image(
    image_path: Path,
) -> tuple[bool, int | None, int | None]:

    try:
        with Image.open(image_path) as image:
            image.verify()

        with Image.open(image_path) as image:
            width, height = image.size

        is_valid = (
            width >= 32
            and height >= 32
        )

        return is_valid, width, height

    except Exception:
        return False, None, None


def download_image(
    row: dict[str, Any],
) -> dict[str, Any]:

    asin = str(
        row["asin"]
    ).strip()

    image_url = str(
        row["imgUrl"]
    ).strip()

    output_path = build_image_path(
        asin=asin,
        image_url=image_url,
    )

    if output_path.exists():

        is_valid, width, height = (
            validate_image(
                output_path
            )
        )

        if is_valid:
            return {
                "asin": asin,
                "imgUrl": image_url,
                "absolute_path": str(
                    output_path
                ),
                "download_success": True,
                "download_status": (
                    "already_exists"
                ),
                "http_status": 200,
                "width": width,
                "height": height,
                "file_size_bytes": (
                    output_path
                    .stat()
                    .st_size
                ),
            }

    headers = {
        "User-Agent": (
            "Mozilla/5.0 "
            "(Macintosh; Intel Mac OS X 10_15_7) "
            "AppleWebKit/537.36 "
            "Chrome/124 Safari/537.36"
        )
    }

    last_error = ""
    last_status = None

    for attempt in range(
        1,
        MAX_RETRIES + 1,
    ):

        try:
            response = requests.get(
                image_url,
                headers=headers,
                timeout=REQUEST_TIMEOUT,
            )

            last_status = (
                response.status_code
            )

            if response.status_code != 200:
                last_error = (
                    f"HTTP "
                    f"{response.status_code}"
                )

                time.sleep(attempt)
                continue

            content_type = (
                response.headers
                .get(
                    "Content-Type",
                    "",
                )
                .lower()
            )

            if "image" not in content_type:
                last_error = (
                    "Invalid content type: "
                    f"{content_type}"
                )

                continue

            output_path.write_bytes(
                response.content
            )

            (
                is_valid,
                width,
                height,
            ) = validate_image(
                output_path
            )

            if not is_valid:
                output_path.unlink(
                    missing_ok=True
                )

                last_error = (
                    "Invalid or corrupt image"
                )

                continue

            return {
                "asin": asin,
                "imgUrl": image_url,
                "absolute_path": str(
                    output_path
                ),
                "download_success": True,
                "download_status": (
                    "downloaded"
                ),
                "http_status": (
                    last_status
                ),
                "width": width,
                "height": height,
                "file_size_bytes": (
                    output_path
                    .stat()
                    .st_size
                ),
            }

        except Exception as error:
            last_error = str(error)
            time.sleep(attempt)

    return {
        "asin": asin,
        "imgUrl": image_url,
        "absolute_path": "",
        "download_success": False,
        "download_status": last_error,
        "http_status": last_status,
        "width": None,
        "height": None,
        "file_size_bytes": None,
    }


# ============================================================
# LOAD SAMPLE
# ============================================================

if not SAMPLE_FILE.exists():
    raise FileNotFoundError(
        f"Sample file not found: "
        f"{SAMPLE_FILE}"
    )

sample_df = pd.read_parquet(
    SAMPLE_FILE
)

print(
    "Products to download:",
    f"{len(sample_df):,}",
)


# ============================================================
# DOWNLOAD IMAGES
# ============================================================

download_rows = sample_df[
    [
        "asin",
        "imgUrl",
    ]
].to_dict(
    orient="records"
)

download_results = []

with concurrent.futures.ThreadPoolExecutor(
    max_workers=MAX_WORKERS
) as executor:

    future_map = {
        executor.submit(
            download_image,
            row,
        ): row["asin"]
        for row in download_rows
    }

    for future in tqdm(
        concurrent.futures.as_completed(
            future_map
        ),
        total=len(future_map),
        desc="Downloading Amazon images",
    ):

        try:
            result = future.result()

        except Exception as error:
            asin = future_map[future]

            result = {
                "asin": asin,
                "imgUrl": "",
                "absolute_path": "",
                "download_success": False,
                "download_status": str(error),
                "http_status": None,
                "width": None,
                "height": None,
                "file_size_bytes": None,
            }

        download_results.append(
            result
        )


# ============================================================
# SAVE DOWNLOAD REPORT
# ============================================================

download_report_df = pd.DataFrame(
    download_results
)

download_report_df.to_csv(
    DOWNLOAD_REPORT_FILE,
    index=False,
)

successful_count = int(
    download_report_df[
        "download_success"
    ].sum()
)

failed_count = (
    len(download_report_df)
    - successful_count
)

print()
print("=" * 80)
print("IMAGE DOWNLOAD SUMMARY")
print("=" * 80)

print(
    "Successful images:",
    f"{successful_count:,}",
)

print(
    "Failed images:",
    f"{failed_count:,}",
)

if failed_count > 0:
    print("\nTop failure reasons:")

    display(
        download_report_df.loc[
            ~download_report_df[
                "download_success"
            ],
            "download_status",
        ]
        .value_counts()
        .head(20)
    )


# ============================================================
# BUILD USABLE MULTIMODAL DATASET
# ============================================================

usable_df = sample_df.merge(
    download_report_df[
        [
            "asin",
            "absolute_path",
            "download_success",
            "download_status",
            "width",
            "height",
            "file_size_bytes",
        ]
    ],
    on="asin",
    how="left",
)

usable_df = usable_df[
    usable_df["download_success"]
    .fillna(False)
].copy()

usable_df["image_exists"] = (
    usable_df["absolute_path"]
    .fillna("")
    .astype(str)
    .map(
        lambda path: (
            bool(path)
            and Path(path).exists()
        )
    )
)

usable_df = usable_df[
    usable_df["image_exists"]
].copy()

usable_df = (
    usable_df
    .drop_duplicates(
        subset=["asin"],
        keep="first",
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# CREATE MODEL TEXT
# ============================================================

usable_df["model_text"] = (
    "title: "
    + usable_df["title"]
    .fillna("")
    .astype(str)
    + " | category: "
    + usable_df["category_name"]
    .fillna("Unknown")
    .astype(str)
)


# ============================================================
# SAVE USABLE DATASET
# ============================================================

usable_df.to_parquet(
    USABLE_DATASET_FILE,
    index=False,
    compression="snappy",
)

print()
print(
    "Usable multimodal rows:",
    f"{len(usable_df):,}",
)

print(
    "Usable dataset saved:",
    USABLE_DATASET_FILE,
)

Products to download: 20,000



IMAGE DOWNLOAD SUMMARY
Successful images: 19,978
Failed images: 22

Top failure reasons:


download_status
Invalid or corrupt image    19
HTTP 404                     3
Name: count, dtype: int64


Usable multimodal rows: 19,978
Usable dataset saved: /Users/souravkumar/Downloads/AI-Powered-Smart-Product-Pricing-Visual-Attribute-Extraction-System/data/amazon_multimodal/model_input/amazon_multimodal_usable.parquet


## create the final train, validation, and test splits from the 19,978

In [19]:
from __future__ import annotations

from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split


# ============================================================
# CONFIGURATION
# ============================================================

CURRENT_DIRECTORY = Path.cwd()

PROJECT_ROOT = (
    CURRENT_DIRECTORY.parent
    if CURRENT_DIRECTORY.name.lower() == "notebooks"
    else CURRENT_DIRECTORY
)

INPUT_FILE = (
    PROJECT_ROOT
    / "data"
    / "amazon_multimodal"
    / "model_input"
    / "amazon_multimodal_usable.parquet"
)

OUTPUT_ROOT = (
    PROJECT_ROOT
    / "data"
    / "amazon_multimodal"
    / "model_input"
)

REPORT_ROOT = (
    PROJECT_ROOT
    / "data"
    / "amazon_multimodal"
    / "reports"
)

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

REPORT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


TRAIN_FILE = (
    OUTPUT_ROOT
    / "train.parquet"
)

VALIDATION_FILE = (
    OUTPUT_ROOT
    / "validation.parquet"
)

TEST_FILE = (
    OUTPUT_ROOT
    / "test.parquet"
)

FULL_SPLIT_FILE = (
    OUTPUT_ROOT
    / "amazon_multimodal_split_dataset.parquet"
)

SPLIT_SUMMARY_FILE = (
    REPORT_ROOT
    / "dataset_split_summary.csv"
)

GROUP_DISTRIBUTION_FILE = (
    REPORT_ROOT
    / "split_group_distribution.csv"
)

CATEGORY_DISTRIBUTION_FILE = (
    REPORT_ROOT
    / "split_category_distribution.csv"
)

PRICE_BAND_DISTRIBUTION_FILE = (
    REPORT_ROOT
    / "split_price_band_distribution.csv"
)


RANDOM_STATE = 42

TRAIN_RATIO = 0.70
VALIDATION_RATIO = 0.15
TEST_RATIO = 0.15

MINIMUM_GROUP_SIZE = 7


# ============================================================
# DISPLAY HELPERS
# ============================================================

def print_header(title: str) -> None:
    print()
    print("=" * 80)
    print(title)
    print("=" * 80)


def print_pass(message: str) -> None:
    print(f"✅ [PASS] {message}")


def print_info(message: str) -> None:
    print(f"ℹ️  [INFO] {message}")


def print_warning(message: str) -> None:
    print(f"⚠️  [WARNING] {message}")


def print_fail(message: str) -> None:
    print(f"❌ [FAIL] {message}")


# ============================================================
# INPUT VALIDATION
# ============================================================

print_header("VALIDATING MULTIMODAL INPUT DATASET")

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Input dataset not found: {INPUT_FILE}"
    )

print_pass(
    f"Input dataset found:\n{INPUT_FILE}"
)


# ============================================================
# LOAD DATASET
# ============================================================

print_header("LOADING MULTIMODAL DATASET")

df = pd.read_parquet(
    INPUT_FILE
)

original_rows = len(df)

print_pass(
    f"Rows loaded: {original_rows:,}"
)

print_info(
    f"Columns available: {len(df.columns):,}"
)


# ============================================================
# REQUIRED COLUMN CHECK
# ============================================================

required_columns = [
    "asin",
    "title",
    "category_name",
    "price",
    "price_band",
    "cluster_id",
    "absolute_path",
    "model_text",
]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        "Required columns are missing: "
        + ", ".join(missing_columns)
    )

print_pass(
    "All required columns are available."
)


# ============================================================
# CLEAN IDENTIFIERS AND TEXT
# ============================================================

print_header("CLEANING SPLIT DATA")

text_columns = [
    "asin",
    "title",
    "category_name",
    "price_band",
    "absolute_path",
    "model_text",
]

for column in text_columns:
    df[column] = (
        df[column]
        .fillna("")
        .astype(str)
        .str.replace(
            r"\s+",
            " ",
            regex=True,
        )
        .str.strip()
    )


df["asin"] = (
    df["asin"]
    .str.upper()
)


df["price"] = pd.to_numeric(
    df["price"],
    errors="coerce",
)

df["cluster_id"] = pd.to_numeric(
    df["cluster_id"],
    errors="coerce",
)


# ============================================================
# FILTER INVALID RECORDS
# ============================================================

invalid_asin_mask = (
    df["asin"].eq("")
)

invalid_title_mask = (
    df["title"].eq("")
)

invalid_price_mask = (
    df["price"].isna()
    | df["price"].le(0)
)

invalid_cluster_mask = (
    df["cluster_id"].isna()
)

invalid_image_path_mask = (
    df["absolute_path"].eq("")
)

invalid_model_text_mask = (
    df["model_text"].eq("")
)


print_info(
    f"Missing ASIN rows: "
    f"{int(invalid_asin_mask.sum()):,}"
)

print_info(
    f"Missing title rows: "
    f"{int(invalid_title_mask.sum()):,}"
)

print_info(
    f"Invalid price rows: "
    f"{int(invalid_price_mask.sum()):,}"
)

print_info(
    f"Missing cluster rows: "
    f"{int(invalid_cluster_mask.sum()):,}"
)

print_info(
    f"Missing image-path rows: "
    f"{int(invalid_image_path_mask.sum()):,}"
)

print_info(
    f"Missing model-text rows: "
    f"{int(invalid_model_text_mask.sum()):,}"
)


valid_mask = ~(
    invalid_asin_mask
    | invalid_title_mask
    | invalid_price_mask
    | invalid_cluster_mask
    | invalid_image_path_mask
    | invalid_model_text_mask
)

df = df[
    valid_mask
].copy()


# ============================================================
# VERIFY IMAGE FILES
# ============================================================

print_header("VERIFYING LOCAL IMAGE FILES")

df["image_file_exists"] = (
    df["absolute_path"]
    .map(
        lambda path: Path(path).exists()
    )
)

missing_image_files = int(
    (~df["image_file_exists"]).sum()
)

if missing_image_files:
    print_warning(
        f"Missing image files on disk: "
        f"{missing_image_files:,}"
    )

df = df[
    df["image_file_exists"]
].copy()

print_pass(
    f"Rows with valid local images: "
    f"{len(df):,}"
)


# ============================================================
# REMOVE DUPLICATE PRODUCTS
# ============================================================

print_header("REMOVING DUPLICATE PRODUCTS")

before_deduplication = len(df)

df = (
    df
    .drop_duplicates(
        subset=["asin"],
        keep="first",
    )
    .reset_index(drop=True)
)

duplicates_removed = (
    before_deduplication
    - len(df)
)

print_pass(
    f"Duplicate ASIN rows removed: "
    f"{duplicates_removed:,}"
)


# ============================================================
# CREATE STRATIFICATION GROUP
# ============================================================

print_header("CREATING STRATIFICATION GROUPS")

df["cluster_id"] = (
    df["cluster_id"]
    .astype(int)
)

df["category_name"] = (
    df["category_name"]
    .replace("", "Unknown")
)

df["price_band"] = (
    df["price_band"]
    .replace("", "Unknown")
)


df["original_split_group"] = (
    "cluster_"
    + df["cluster_id"].astype(str)
    + "__price_"
    + df["price_band"]
)


original_group_counts = (
    df["original_split_group"]
    .value_counts()
)

rare_groups = original_group_counts[
    original_group_counts
    < MINIMUM_GROUP_SIZE
].index


df["split_group"] = (
    df["original_split_group"]
)

df.loc[
    df["split_group"].isin(
        rare_groups
    ),
    "split_group",
] = (
    "cluster_"
    + df.loc[
        df["split_group"].isin(
            rare_groups
        ),
        "cluster_id",
    ].astype(str)
    + "__rare_price"
)


# If the cluster-level fallback is still too small,
# merge it into one global rare group.

fallback_group_counts = (
    df["split_group"]
    .value_counts()
)

still_rare_groups = fallback_group_counts[
    fallback_group_counts
    < MINIMUM_GROUP_SIZE
].index

df.loc[
    df["split_group"].isin(
        still_rare_groups
    ),
    "split_group",
] = "global_rare_group"


final_group_counts = (
    df["split_group"]
    .value_counts()
)

print_info(
    f"Original stratification groups: "
    f"{len(original_group_counts):,}"
)

print_info(
    f"Final stratification groups: "
    f"{len(final_group_counts):,}"
)

print_info(
    f"Smallest final group: "
    f"{final_group_counts.min():,}"
)

print_info(
    f"Largest final group: "
    f"{final_group_counts.max():,}"
)


# ============================================================
# CREATE TRAIN AND TEMPORARY SPLIT
# ============================================================

print_header("CREATING TRAIN SPLIT")

temporary_ratio = (
    VALIDATION_RATIO
    + TEST_RATIO
)

train_df, temporary_df = train_test_split(
    df,
    test_size=temporary_ratio,
    random_state=RANDOM_STATE,
    shuffle=True,
    stratify=df["split_group"],
)

print_pass(
    f"Training rows created: "
    f"{len(train_df):,}"
)

print_pass(
    f"Temporary rows created: "
    f"{len(temporary_df):,}"
)


# ============================================================
# CREATE VALIDATION AND TEST SPLITS
# ============================================================

print_header("CREATING VALIDATION AND TEST SPLITS")

validation_fraction_of_temporary = (
    VALIDATION_RATIO
    / temporary_ratio
)

temporary_group_counts = (
    temporary_df["split_group"]
    .value_counts()
)

temporary_rare_groups = (
    temporary_group_counts[
        temporary_group_counts < 2
    ].index
)

temporary_df = temporary_df.copy()

temporary_df.loc[
    temporary_df["split_group"].isin(
        temporary_rare_groups
    ),
    "temporary_split_group",
] = "temporary_rare_group"

temporary_df.loc[
    ~temporary_df["split_group"].isin(
        temporary_rare_groups
    ),
    "temporary_split_group",
] = temporary_df.loc[
    ~temporary_df["split_group"].isin(
        temporary_rare_groups
    ),
    "split_group",
]


validation_df, test_df = train_test_split(
    temporary_df,
    test_size=(
        1 - validation_fraction_of_temporary
    ),
    random_state=RANDOM_STATE,
    shuffle=True,
    stratify=temporary_df[
        "temporary_split_group"
    ],
)

print_pass(
    f"Validation rows created: "
    f"{len(validation_df):,}"
)

print_pass(
    f"Test rows created: "
    f"{len(test_df):,}"
)


# ============================================================
# ASSIGN DATASET LABELS
# ============================================================

train_df = train_df.copy()
validation_df = validation_df.copy()
test_df = test_df.copy()

train_df["dataset_split"] = "train"

validation_df[
    "dataset_split"
] = "validation"

test_df["dataset_split"] = "test"


# ============================================================
# LEAKAGE CHECK
# ============================================================

print_header("CHECKING DATA LEAKAGE")

train_asins = set(
    train_df["asin"]
)

validation_asins = set(
    validation_df["asin"]
)

test_asins = set(
    test_df["asin"]
)


train_validation_overlap = (
    train_asins
    & validation_asins
)

train_test_overlap = (
    train_asins
    & test_asins
)

validation_test_overlap = (
    validation_asins
    & test_asins
)


if (
    train_validation_overlap
    or train_test_overlap
    or validation_test_overlap
):
    raise RuntimeError(
        "ASIN leakage was detected between splits."
    )

print_pass(
    "No ASIN leakage detected."
)


# ============================================================
# COVERAGE CHECKS
# ============================================================

print_header("CHECKING CLUSTER AND PRICE-BAND COVERAGE")


def print_split_coverage(
    split_name: str,
    split_df: pd.DataFrame,
) -> None:

    print(
        f"{split_name}: "
        f"{len(split_df):,} rows | "
        f"{split_df['cluster_id'].nunique():,} clusters | "
        f"{split_df['price_band'].nunique():,} price bands | "
        f"{split_df['category_name'].nunique():,} categories"
    )


print_split_coverage(
    "Train",
    train_df,
)

print_split_coverage(
    "Validation",
    validation_df,
)

print_split_coverage(
    "Test",
    test_df,
)


# ============================================================
# DROP TEMPORARY SPLIT COLUMNS
# ============================================================

temporary_columns = [
    "temporary_split_group",
]

for split_df in [
    train_df,
    validation_df,
    test_df,
]:
    split_df.drop(
        columns=[
            column
            for column in temporary_columns
            if column in split_df.columns
        ],
        inplace=True,
    )


# ============================================================
# SAVE SPLIT FILES
# ============================================================

print_header("SAVING DATASET SPLITS")

train_df.to_parquet(
    TRAIN_FILE,
    index=False,
    compression="snappy",
)

validation_df.to_parquet(
    VALIDATION_FILE,
    index=False,
    compression="snappy",
)

test_df.to_parquet(
    TEST_FILE,
    index=False,
    compression="snappy",
)


full_split_df = pd.concat(
    [
        train_df,
        validation_df,
        test_df,
    ],
    ignore_index=True,
)

full_split_df.to_parquet(
    FULL_SPLIT_FILE,
    index=False,
    compression="snappy",
)


print_pass(
    f"Training dataset saved:\n"
    f"{TRAIN_FILE}"
)

print_pass(
    f"Validation dataset saved:\n"
    f"{VALIDATION_FILE}"
)

print_pass(
    f"Test dataset saved:\n"
    f"{TEST_FILE}"
)

print_pass(
    f"Complete split dataset saved:\n"
    f"{FULL_SPLIT_FILE}"
)


# ============================================================
# CREATE SUMMARY REPORT
# ============================================================

print_header("CREATING SPLIT SUMMARY")

summary_df = pd.DataFrame(
    [
        {
            "metric": "Original usable rows",
            "value": original_rows,
        },
        {
            "metric": "Final usable rows",
            "value": len(full_split_df),
        },
        {
            "metric": "Duplicate ASINs removed",
            "value": duplicates_removed,
        },
        {
            "metric": "Missing local images removed",
            "value": missing_image_files,
        },
        {
            "metric": "Train rows",
            "value": len(train_df),
        },
        {
            "metric": "Validation rows",
            "value": len(validation_df),
        },
        {
            "metric": "Test rows",
            "value": len(test_df),
        },
        {
            "metric": "Train percentage",
            "value": (
                len(train_df)
                / len(full_split_df)
                * 100
            ),
        },
        {
            "metric": "Validation percentage",
            "value": (
                len(validation_df)
                / len(full_split_df)
                * 100
            ),
        },
        {
            "metric": "Test percentage",
            "value": (
                len(test_df)
                / len(full_split_df)
                * 100
            ),
        },
        {
            "metric": "Minimum price",
            "value": full_split_df[
                "price"
            ].min(),
        },
        {
            "metric": "Median price",
            "value": full_split_df[
                "price"
            ].median(),
        },
        {
            "metric": "Maximum price",
            "value": full_split_df[
                "price"
            ].max(),
        },
    ]
)

summary_df.to_csv(
    SPLIT_SUMMARY_FILE,
    index=False,
)

display(summary_df)


# ============================================================
# GROUP DISTRIBUTION REPORT
# ============================================================

group_distribution_df = (
    full_split_df
    .groupby(
        [
            "dataset_split",
            "split_group",
        ],
        dropna=False,
    )
    .size()
    .reset_index(
        name="product_count"
    )
)

group_distribution_df.to_csv(
    GROUP_DISTRIBUTION_FILE,
    index=False,
)


# ============================================================
# CATEGORY DISTRIBUTION REPORT
# ============================================================

category_distribution_df = (
    full_split_df
    .groupby(
        [
            "dataset_split",
            "category_name",
        ],
        dropna=False,
    )
    .size()
    .reset_index(
        name="product_count"
    )
)

category_distribution_df.to_csv(
    CATEGORY_DISTRIBUTION_FILE,
    index=False,
)


# ============================================================
# PRICE-BAND DISTRIBUTION REPORT
# ============================================================

price_band_distribution_df = (
    full_split_df
    .groupby(
        [
            "dataset_split",
            "price_band",
        ],
        dropna=False,
    )
    .size()
    .reset_index(
        name="product_count"
    )
)

price_band_distribution_df.to_csv(
    PRICE_BAND_DISTRIBUTION_FILE,
    index=False,
)

display(
    price_band_distribution_df
)


# ============================================================
# FINAL FILE VERIFICATION
# ============================================================

print_header("VERIFYING SAVED SPLIT FILES")

for split_name, file_path in [
    ("train", TRAIN_FILE),
    ("validation", VALIDATION_FILE),
    ("test", TEST_FILE),
]:

    saved_df = pd.read_parquet(
        file_path,
        columns=[
            "asin",
            "dataset_split",
        ],
    )

    print_pass(
        f"{split_name}: "
        f"{len(saved_df):,} rows saved successfully"
    )


# ============================================================
# FINAL RESULT
# ============================================================

print_header("TRAIN / VALIDATION / TEST SPLIT COMPLETED")

print(
    f"Train rows: "
    f"{len(train_df):,}"
)

print(
    f"Validation rows: "
    f"{len(validation_df):,}"
)

print(
    f"Test rows: "
    f"{len(test_df):,}"
)

print()
print_pass(
    "Dataset splitting completed without ASIN leakage."
)


VALIDATING MULTIMODAL INPUT DATASET
✅ [PASS] Input dataset found:
/Users/souravkumar/Downloads/AI-Powered-Smart-Product-Pricing-Visual-Attribute-Extraction-System/data/amazon_multimodal/model_input/amazon_multimodal_usable.parquet

LOADING MULTIMODAL DATASET
✅ [PASS] Rows loaded: 19,978
ℹ️  [INFO] Columns available: 22
✅ [PASS] All required columns are available.

CLEANING SPLIT DATA
ℹ️  [INFO] Missing ASIN rows: 0
ℹ️  [INFO] Missing title rows: 0
ℹ️  [INFO] Invalid price rows: 0
ℹ️  [INFO] Missing cluster rows: 0
ℹ️  [INFO] Missing image-path rows: 0
ℹ️  [INFO] Missing model-text rows: 0

VERIFYING LOCAL IMAGE FILES
✅ [PASS] Rows with valid local images: 19,978

REMOVING DUPLICATE PRODUCTS
✅ [PASS] Duplicate ASIN rows removed: 0

CREATING STRATIFICATION GROUPS
ℹ️  [INFO] Original stratification groups: 443
ℹ️  [INFO] Final stratification groups: 432
ℹ️  [INFO] Smallest final group: 7
ℹ️  [INFO] Largest final group: 172

CREATING TRAIN SPLIT
✅ [PASS] Training rows created: 13,984
✅ [P

,metric,value
0,Original usable rows,19978.000000
1,Final usable rows,19978.000000
2,Duplicate ASINs removed,0.000000
3,Missing local images removed,0.000000
4,Train rows,13984.000000
5,Validation rows,2997.000000
6,Test rows,2997.000000
7,Train percentage,69.996997
8,Validation percentage,15.001502
9,Test percentage,15.001502


,dataset_split,price_band,product_count
0,test,high,603
1,test,low,558
2,test,luxury,214
3,test,medium,589
4,test,premium,524
5,test,very_low,509
6,train,high,2813
7,train,low,2600
8,train,luxury,1003
9,train,medium,2741



VERIFYING SAVED SPLIT FILES
✅ [PASS] train: 13,984 rows saved successfully
✅ [PASS] validation: 2,997 rows saved successfully
✅ [PASS] test: 2,997 rows saved successfully

TRAIN / VALIDATION / TEST SPLIT COMPLETED
Train rows: 13,984
Validation rows: 2,997
Test rows: 2,997

✅ [PASS] Dataset splitting completed without ASIN leakage.


In [20]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

for path in PROJECT_ROOT.rglob("train.parquet"):
    print(path)

/Users/souravkumar/Downloads/AI-Powered-Smart-Product-Pricing-Visual-Attribute-Extraction-System/data/model_input/train.parquet
/Users/souravkumar/Downloads/AI-Powered-Smart-Product-Pricing-Visual-Attribute-Extraction-System/data/amazon_multimodal/model_input/train.parquet


In [21]:
for path in PROJECT_ROOT.rglob("validation.parquet"):
    print(path)

/Users/souravkumar/Downloads/AI-Powered-Smart-Product-Pricing-Visual-Attribute-Extraction-System/data/model_input/validation.parquet
/Users/souravkumar/Downloads/AI-Powered-Smart-Product-Pricing-Visual-Attribute-Extraction-System/data/amazon_multimodal/model_input/validation.parquet


In [22]:
for path in PROJECT_ROOT.rglob("test.parquet"):
    print(path)

/Users/souravkumar/Downloads/AI-Powered-Smart-Product-Pricing-Visual-Attribute-Extraction-System/data/model_input/test.parquet
/Users/souravkumar/Downloads/AI-Powered-Smart-Product-Pricing-Visual-Attribute-Extraction-System/data/amazon_multimodal/model_input/test.parquet


In [23]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

for file in PROJECT_ROOT.rglob("*.parquet"):
    size_mb = file.stat().st_size / (1024 * 1024)
    print(f"{size_mb:8.2f} MB   {file}")

    0.00 MB   /Users/souravkumar/Downloads/AI-Powered-Smart-Product-Pricing-Visual-Attribute-Extraction-System/.venv/lib/python3.11/site-packages/pyarrow/tests/data/parquet/v0.7.1.all-named-index.parquet
    0.00 MB   /Users/souravkumar/Downloads/AI-Powered-Smart-Product-Pricing-Visual-Attribute-Extraction-System/.venv/lib/python3.11/site-packages/pyarrow/tests/data/parquet/v0.7.1.column-metadata-handling.parquet
    0.00 MB   /Users/souravkumar/Downloads/AI-Powered-Smart-Product-Pricing-Visual-Attribute-Extraction-System/.venv/lib/python3.11/site-packages/pyarrow/tests/data/parquet/v0.7.1.some-named-index.parquet
    0.00 MB   /Users/souravkumar/Downloads/AI-Powered-Smart-Product-Pricing-Visual-Attribute-Extraction-System/.venv/lib/python3.11/site-packages/pyarrow/tests/data/parquet/v0.7.1.parquet
    0.07 MB   /Users/souravkumar/Downloads/AI-Powered-Smart-Product-Pricing-Visual-Attribute-Extraction-System/data/embeddings/clip/train_metadata.parquet
    0.02 MB   /Users/souravkumar/Do